# Group A ASR Dry Run
This notebook runs a smoke test for the HW14 Group A speech workflow. It is intended for VS Code plus the Colab extension on an A100 or H100 before the full notebook is executed.

Expected dry-run outputs:
- 5 baseline ASR rows
- 4 Experiment 2 short-form ASR rows
- 3 Experiment 3A resampling rows for one utterance and one workflow
- 1 Experiment 3B raw-versus-cleaned GA20C comparison row
- 1 Experiment 4 long-form Whisper row

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%pip install -q transformers datasets accelerate genaibook jiwer scipy soundfile numpy pandas matplotlib sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 153.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 60.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.8 MB/s eta 0:00:00


In [8]:
from pathlib import Path

import os

import sys



DRIVE_ROOT = Path('/content/drive/MyDrive')

CANDIDATE_ROOTS = [

    DRIVE_ROOT / 'kamp_hw14',

    DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw14',

    DRIVE_ROOT / 'GEN_AI' / 'HW1' / 'kamp_hw14',

]



def find_project_root():

    checked_paths = []

    for candidate in CANDIDATE_ROOTS:

        checked_paths.append(candidate)

        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():

            return candidate



    discovered_roots = []

    for match in DRIVE_ROOT.rglob('kamp_hw14.py'):

        if match.parent.name != 'hw14_scripts':

            continue

        candidate = match.parent.parent

        discovered_roots.append(candidate)

        if (candidate / 'hw14_scripts' / 'kamp_hw14.py').exists():

            return candidate



    checked_display = '\n'.join(f' - {path}' for path in checked_paths)

    discovered_display = '\n'.join(f' - {path}' for path in discovered_roots[:10]) or ' - none found'

    raise FileNotFoundError(

        'Could not locate kamp_hw14 on Google Drive.\n'

        'Upload the project to /content/drive/MyDrive/kamp_hw14\n'

        'or keep any Drive copy that contains hw14_scripts/kamp_hw14.py.\n'

        f'Checked:\n{checked_display}\n'

        f'Discovered matches:\n{discovered_display}'

    )



PROJECT_ROOT = find_project_root()

os.chdir(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT / 'hw14_scripts'))



print(f'PROJECT_ROOT = {PROJECT_ROOT}')

print(f'Module exists: {(PROJECT_ROOT / "hw14_scripts" / "kamp_hw14.py").exists()}')

PROJECT_ROOT = /content/drive/MyDrive/kamp_hw14
Module exists: True


In [13]:
from contextlib import redirect_stderr, redirect_stdout

import importlib



import kamp_hw14

kamp_hw14 = importlib.reload(kamp_hw14)



from kamp_hw14 import run_group_a_asr

from hw14_data_utils import CHECKPOINT_PATH, TeeLogger, load_checkpoint, save_checkpoint, save_text_artifact



print(f'kamp_hw14 imported from: {kamp_hw14.__file__}')

print(f'Checkpoint path: {CHECKPOINT_PATH}')

kamp_hw14 imported from: /content/drive/MyDrive/kamp_hw14/hw14_scripts/kamp_hw14.py
Checkpoint path: /content/drive/MyDrive/kamp_hw14/hw14_experiments/checkpoint.json


In [15]:
DRY_RUN_ROOT = PROJECT_ROOT / 'hw14_experiments' / 'group_a_asr' / 'dry_run'
DRY_RUN_LOG = PROJECT_ROOT / 'hw14_printouts' / 'group_a_asr_dry_run_log.txt'
DRY_RUN_SUMMARY_PATH = DRY_RUN_ROOT / 'dry_run_summary.json'
DRY_RUN_CHECKPOINT_SNAPSHOT = DRY_RUN_ROOT / 'dry_run_checkpoint.json'

DRY_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRY_RUN_LOG.parent.mkdir(parents=True, exist_ok=True)

print(f'DRY_RUN_ROOT = {DRY_RUN_ROOT}')
print(f'DRY_RUN_LOG = {DRY_RUN_LOG}')

DRY_RUN_ROOT = /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_a_asr/dry_run
DRY_RUN_LOG = /content/drive/MyDrive/kamp_hw14/hw14_printouts/group_a_asr_dry_run_log.txt


In [16]:
summary = None

with TeeLogger(DRY_RUN_LOG) as tee, redirect_stdout(tee), redirect_stderr(tee):

    print('[NOTEBOOK] Starting Group A dry run')

    summary = run_group_a_asr(mode='dry_run')

    print(summary)



save_text_artifact(DRY_RUN_SUMMARY_PATH, summary, as_json=True)

print(summary)

[NOTEBOOK] Starting Group A dry run
Device set to use cuda:0
[timer] ga20a_baseline: 0.821s
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[timer] ga20b_baseline: 0.153s
[timer] ga20c_baseline: 2.066s
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[timer] ga20d_corrected_baseline: 13.254s
[timer] ga2

In [17]:
current_checkpoint = load_checkpoint(CHECKPOINT_PATH)
save_checkpoint(current_checkpoint, DRY_RUN_CHECKPOINT_SNAPSHOT)
print(f'Saved dry-run checkpoint snapshot: {DRY_RUN_CHECKPOINT_SNAPSHOT}')

Saved dry-run checkpoint snapshot: /content/drive/MyDrive/kamp_hw14/hw14_experiments/group_a_asr/dry_run/dry_run_checkpoint.json


In [18]:
assert summary['counts']['baseline_rows'] == 5, summary['counts']
assert summary['counts']['exp2_rows'] == 4, summary['counts']
assert summary['counts']['exp3_rows'] == 4, summary['counts']
assert summary['counts']['exp4_rows'] == 1, summary['counts']

assert (DRY_RUN_ROOT / 'exp1_baselines' / 'exp1_baselines.csv').exists()
assert (DRY_RUN_ROOT / 'exp2_shortform_asr' / 'exp2_shortform_asr.csv').exists()
assert (DRY_RUN_ROOT / 'exp3_asr_sensitivity' / 'exp3_asr_sensitivity.csv').exists()
assert (DRY_RUN_ROOT / 'exp4_longform_whisper' / 'exp4_longform_whisper.csv').exists()
assert len(list((DRY_RUN_ROOT / 'exp4_longform_whisper').glob('*.json'))) >= 1
assert DRY_RUN_LOG.exists()
assert DRY_RUN_SUMMARY_PATH.exists()
assert DRY_RUN_CHECKPOINT_SNAPSHOT.exists()
print('Dry-run smoke test completed successfully.')

Dry-run smoke test completed successfully.
